# OmniServe — QLoRA fine-tuning on ColabTrains the domain SLM from the [OmniServe](https://github.com/JCHETAN26/Omni-Serve)build plan and produces the before/after numbers Phase 7 reports.**Runtime → Change runtime type → T4 GPU** (or L4/A100 if you have Pro).What this notebook does, in order:1. Install dependencies and clone the repo2. Generate the synthetic dataset and split it 8500 / 1000 / 5003. **Measure the untuned baseline** — this is the number the whole project is compared against4. Fine-tune with QLoRA5. Measure the tuned model on the *same* test records6. Save the adapter to Google DriveSteps 3 and 5 use identical prompts and scoring code, so the comparison is honest.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 1. InstallUnsloth pins its own torch/transformers combination, so let it drive the install.Expect ~3 minutes and one restart prompt you can ignore.

In [ ]:
%%capture!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
!git clone -q https://github.com/JCHETAN26/Omni-Serve.git omniserve%cd omniserve!pip install -q -e .

## 2. Dataset`--offline` renders documents from templates: free, deterministic, ~2 seconds for 10k.To use GPT-4o rendering instead, set `OPENAI_API_KEY` and drop `--offline`. Notethat GPT-4o only writes the *document text* — the JSON labels are generatedprogrammatically either way, so they are correct by construction.

In [ ]:
!python -m data.generate_dataset --count 10000 --offline --noise 0.01!python -m data.split_dataset

In [ ]:
import jsonrecord = json.loads(open("data/generated/train.jsonl").readline())print(record["text"][:400])print("\n--- target ---")print(json.dumps(record["target"], indent=2)[:300])

## 3. Baseline — the untuned modelRun this **before** training. Afterwards the base weights are still on disk, butmeasuring first means you cannot accidentally report a baseline that wasinfluenced by anything you did later.The baseline gets `--include-schema`, which puts the full JSON schema in theprompt. It needs that to have any idea what fields to produce; the fine-tunedmodel won't get it. That asymmetry is deliberate and favours the baseline, sothe improvement you measure is understated rather than inflated.`--limit 200` keeps this to roughly 10-15 minutes on a T4. Drop it for the full500-record number when you're producing final results.

In [ ]:
!python -m benchmarks.eval_local \    --model unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit \    --tag baseline --include-schema \    --batch-size 8 --limit 200

## 4. Fine-tune`--max-seq-length 1024`: the longest training example in this dataset is ~640tokens, so 1024 has comfortable headroom. The `TrainConfig` default of 4096 is4× that and will OOM an 8B on a 16GB T4 for no benefit.`--epochs 1` on a free T4 (~50-70 min). Free Colab disconnects around 90minutes of inactivity, so 2 epochs is realistic only on L4/A100 — bump it if youhave one.The model is Unsloth's 4-bit mirror rather than `meta-llama/...`, which avoidsthe gated-repo prompt and needs no HF token.

In [ ]:
!python -m training.train_qlora \    --model unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit \    --max-seq-length 1024 \    --epochs 1 \    --output training/adapters/omniserve-slm-8b

## 5. Tuned model — same test recordsNo `--include-schema` here: the fine-tune learned the schema, and omitting itsaves ~400 tokens of prefill per request at serve time. Keep `--limit` identicalto the baseline cell or the two numbers aren't comparable.

In [ ]:
!python -m benchmarks.eval_local \    --model unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit \    --adapter training/adapters/omniserve-slm-8b \    --tag tuned \    --batch-size 8 --limit 200

## 6. Compare

In [ ]:
import jsonfrom pathlib import Pathbaseline = json.loads(Path("benchmarks/results/accuracy-baseline.json").read_text())tuned = json.loads(Path("benchmarks/results/accuracy-tuned.json").read_text())rows = [    ("Field F1", "field_f1"),    ("Field precision", "field_precision"),    ("Field recall", "field_recall"),    ("Exact match rate", "exact_match_rate"),    ("Schema validity rate", "schema_validity_rate"),    ("Invalid JSON syntax rate", "invalid_json_syntax_rate"),]print(f"{'metric':<26}{'baseline':>10}{'tuned':>10}{'delta':>10}")print("-" * 56)for label, key in rows:    b, t = baseline[key], tuned[key]    print(f"{label:<26}{b:>10.4f}{t:>10.4f}{t - b:>+10.4f}")print()print("Baseline saw the schema in its prompt; the tuned model did not.")print("Constrained decoding is not active here — it lands at serve time (Phase 4)")print("and takes schema validity to 1.0 on its own. What this table measures is")print("what fine-tuning contributed, which is the field-level accuracy.")

## 7. Save the adapterThe adapter is ~160MB. Colab wipes local disk when the runtime ends, so copy itsomewhere durable before closing the tab.

In [ ]:
from google.colab import drivedrive.mount("/content/drive")!mkdir -p "/content/drive/MyDrive/omniserve"!cp -r training/adapters/omniserve-slm-8b "/content/drive/MyDrive/omniserve/"!cp benchmarks/results/*.json "/content/drive/MyDrive/omniserve/"!du -sh "/content/drive/MyDrive/omniserve/omniserve-slm-8b"

### Optional: push to the Hugging Face HubHandy if you want vLLM to pull the adapter by name later instead of from a path.

In [ ]:
# from huggingface_hub import notebook_login# notebook_login()## from peft import PeftModel# PeftModel.from_pretrained(model, "training/adapters/omniserve-slm-8b").push_to_hub(#     "your-username/omniserve-slm-8b"# )

## NextServing happens off Colab — vLLM wants a persistent GPU, not a notebook:```bashpython -m gateway.main --model-path ./training/adapters/omniserve-slm-8b```Copy `benchmarks/results/*.json` back into the repo so Phase 7 can chart thebefore/after numbers against the load-test results.